In [2]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

In [1]:
# import the required packages
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from tqdm.auto import tqdm
from pprint import pprint
import numpy as np
import os

from ingest import load_faq_data, build_vector_index
from rag_helper import RAGVector

In [ ]:
# load the environment variables
load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
hf_token = os.getenv('HF_TOKEN')

llm_client = OpenAI(api_key=api_key)
model = SentenceTransformer(model_name_or_path='all-MiniLM-L6-v2', token=hf_token)

documents = load_faq_data()

texts = []
vectors = []
batch_size = 50

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)
vindex = build_vector_index(X=X, documents=documents)
print('Vector Index created.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

Vector Index created.


In [ ]:
vector_assistant = RAGVector(
    embedder=model, 
    index=vindex,
    llm_client=llm_client
)

answer = vector_assistant.rag("the program has already begun, can I still sign up?")
print(answer)

Yes, you can still join. You don’t need a confirmation email, and you can start learning and submitting homework while the form is open.
